# NLP Text Preprocessing Project

### Tools: Python, NLTK, Regex

This notebook demonstrates a basic NLP preprocessing pipeline on three different kinds of text: a social-media post, a news-style article excerpt, and a customer review.

The project covers five main transformation steps:
1. Sentence-level and word-level tokenisation
2. Stopword removal
3. Punctuation stripping with Regex
4. Stemming with `PorterStemmer`
5. Lemmatization with `WordNetLemmatizer`

For every important step, the notebook prints a before/after comparison and explains why the step matters.

## 1. Project objectives

The goal is not simply to clean text, but to understand **what each preprocessing operation changes** and **why that change can help downstream NLP tasks** such as text classification, sentiment analysis, search, clustering, and machine learning.

> **Important source note:** The three examples below are short, original, genre-representative samples so the notebook is self-contained. If your instructor strictly requires real externally sourced text, replace the three strings with short excerpts from your chosen social-media post, news article, and customer review, and record the URLs/source names in the `source` field.

## 2. Import the required libraries

We first import the Python and NLTK tools needed by the project.

In [1]:
import re
import nltk

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

### What each new line does

- `import re` loads Python's **regular-expression** module. We use it later to remove punctuation.
- `import nltk` loads the Natural Language Toolkit.
- `word_tokenize` splits text into individual word/punctuation tokens.
- `sent_tokenize` splits a paragraph into sentences.
- `stopwords` gives us NLTK's built-in list of common words such as `the`, `is`, `and`, and `to`.
- `PorterStemmer` reduces words to stems, often by cutting off endings.
- `WordNetLemmatizer` reduces words to dictionary-style base forms using WordNet.

## 3. Download the NLTK resources

NLTK's algorithms are installed with the package, but some language data is downloaded separately. The following commands make the notebook work with current NLTK installations.

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Abc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Abc\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Abc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Abc\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Abc\AppData\Roaming\nltk_data...


True

### Why are these downloads necessary?

- `punkt` and `punkt_tab` provide sentence/word tokenisation data used by NLTK tokenizers.
- `stopwords` provides the English stopword corpus.
- `wordnet` provides the lexical database needed by `WordNetLemmatizer`.
- `omw-1.4` supplies additional WordNet language information used by NLTK's WordNet interface.

If NLTK says a resource is already up to date, that is normal.

## 4. Load the three raw text samples

The samples deliberately contain punctuation, emojis, stopwords, and inconsistent casing so that the preprocessing steps have visible effects.

In [3]:
samples = {
    'Social Media': {
        'source': 'Original representative social-media sample',
        'text': 'OMG!!! I LOVE this new phone update 😍🔥 It is SO fast now, but the battery drains quickly... #happy'
    },
    'News Article': {
        'source': 'Original representative news-style sample',
        'text': 'The city council announced that the new transit line will open next month after months of testing. Officials said the service will reduce traffic and improve access.'
    },
    'Customer Review': {
        'source': 'Original representative customer-review sample',
        'text': 'The room was clean, the staff were friendly, and the breakfast was amazing! I would definitely stay here again 😊.'
    }
}

for sample_name, sample in samples.items():
    print(f'--- {sample_name} ---')
    print('Source:', sample['source'])
    print('Raw text:', sample['text'])
    print()

--- Social Media ---
Source: Original representative social-media sample
Raw text: OMG!!! I LOVE this new phone update 😍🔥 It is SO fast now, but the battery drains quickly... #happy

--- News Article ---
Source: Original representative news-style sample
Raw text: The city council announced that the new transit line will open next month after months of testing. Officials said the service will reduce traffic and improve access.

--- Customer Review ---
Source: Original representative customer-review sample
Raw text: The room was clean, the staff were friendly, and the breakfast was amazing! I would definitely stay here again 😊.



### Why keep the raw text?

The raw text is our **baseline**. We should never overwrite it because every later transformation needs something to compare against. Keeping the original also makes the preprocessing pipeline reproducible and easier to debug.

## 5. Step 1 — Sentence-level tokenisation

Sentence tokenisation identifies where one sentence ends and another begins. This is useful when an NLP task needs to process text sentence by sentence.

In [4]:
sentence_tokens = {}

for sample_name, sample in samples.items():
    sentences = sent_tokenize(sample['text'])
    sentence_tokens[sample_name] = sentences

    print(f'--- {sample_name} ---')
    print('BEFORE:', sample['text'])
    print('AFTER :', sentences)
    print('Number of sentences:', len(sentences))
    print()

--- Social Media ---
BEFORE: OMG!!! I LOVE this new phone update 😍🔥 It is SO fast now, but the battery drains quickly... #happy
AFTER : ['OMG!!!', 'I LOVE this new phone update 😍🔥 It is SO fast now, but the battery drains quickly... #happy']
Number of sentences: 2

--- News Article ---
BEFORE: The city council announced that the new transit line will open next month after months of testing. Officials said the service will reduce traffic and improve access.
AFTER : ['The city council announced that the new transit line will open next month after months of testing.', 'Officials said the service will reduce traffic and improve access.']
Number of sentences: 2

--- Customer Review ---
BEFORE: The room was clean, the staff were friendly, and the breakfast was amazing! I would definitely stay here again 😊.
AFTER : ['The room was clean, the staff were friendly, and the breakfast was amazing!', 'I would definitely stay here again 😊.']
Number of sentences: 2



### New commands explained

- `sent_tokenize(text)` examines the text and returns a Python list containing the detected sentences.
- `sentence_tokens = {}` creates an empty dictionary where we store the result for each sample.
- `for sample_name, sample in samples.items():` loops through every sample in our dictionary.
- `len(sentences)` counts how many sentences were found.

### Why sentence tokenisation matters

If we skip it, a model that expects sentence-level input may receive a whole paragraph as one unit. This can make sentence sentiment, summarisation, question answering, and sentence-level classification less precise.

## 6. Step 1 continued — Word-level tokenisation

Word tokenisation breaks each sample into smaller tokens. Notice that punctuation and emoji-like symbols can appear as separate tokens; this is intentional because tokenisation should first identify pieces of text before later cleaning operations remove unwanted pieces.

In [5]:
word_tokens = {}

for sample_name, sample in samples.items():
    tokens = word_tokenize(sample['text'])
    word_tokens[sample_name] = tokens

    print(f'--- {sample_name} ---')
    print('BEFORE:', sample['text'])
    print('AFTER :', tokens)
    print('Number of word-level tokens:', len(tokens))
    print()

--- Social Media ---
BEFORE: OMG!!! I LOVE this new phone update 😍🔥 It is SO fast now, but the battery drains quickly... #happy
AFTER : ['OMG', '!', '!', '!', 'I', 'LOVE', 'this', 'new', 'phone', 'update', '😍🔥', 'It', 'is', 'SO', 'fast', 'now', ',', 'but', 'the', 'battery', 'drains', 'quickly', '...', '#', 'happy']
Number of word-level tokens: 25

--- News Article ---
BEFORE: The city council announced that the new transit line will open next month after months of testing. Officials said the service will reduce traffic and improve access.
AFTER : ['The', 'city', 'council', 'announced', 'that', 'the', 'new', 'transit', 'line', 'will', 'open', 'next', 'month', 'after', 'months', 'of', 'testing', '.', 'Officials', 'said', 'the', 'service', 'will', 'reduce', 'traffic', 'and', 'improve', 'access', '.']
Number of word-level tokens: 29

--- Customer Review ---
BEFORE: The room was clean, the staff were friendly, and the breakfast was amazing! I would definitely stay here again 😊.
AFTER : ['Th

### Why word tokenisation matters

Most classical NLP operations work on tokens rather than an entire paragraph. Without tokenisation, we cannot reliably count words, remove stopwords, calculate vocabulary size, stem words, or lemmatise them.

## 7. Step 2 — Stopword removal

Stopwords are very common grammatical words that often contribute little to a basic bag-of-words representation. We use NLTK's built-in English stopword corpus.

We compare vocabulary size **before and after** stopword removal. For matching, we use `token.lower()` so that words such as `The` and `the` are treated consistently, while the original token is retained in the output.

In [6]:
stop_words = set(stopwords.words('english'))
no_stopwords = {}

for sample_name, tokens in word_tokens.items():
    before_vocab = set(token.lower() for token in tokens)

    filtered_tokens = [
        token for token in tokens
        if token.lower() not in stop_words
    ]
    no_stopwords[sample_name] = filtered_tokens

    after_vocab = set(token.lower() for token in filtered_tokens)

    print(f'--- {sample_name} ---')
    print('BEFORE:', tokens)
    print('AFTER :', filtered_tokens)
    print('Vocabulary size before:', len(before_vocab))
    print('Vocabulary size after :', len(after_vocab))
    print('Vocabulary reduction   :', len(before_vocab) - len(after_vocab))
    print()

--- Social Media ---
BEFORE: ['OMG', '!', '!', '!', 'I', 'LOVE', 'this', 'new', 'phone', 'update', '😍🔥', 'It', 'is', 'SO', 'fast', 'now', ',', 'but', 'the', 'battery', 'drains', 'quickly', '...', '#', 'happy']
AFTER : ['OMG', '!', '!', '!', 'LOVE', 'new', 'phone', 'update', '😍🔥', 'fast', ',', 'battery', 'drains', 'quickly', '...', '#', 'happy']
Vocabulary size before: 23
Vocabulary size after : 15
Vocabulary reduction   : 8

--- News Article ---
BEFORE: ['The', 'city', 'council', 'announced', 'that', 'the', 'new', 'transit', 'line', 'will', 'open', 'next', 'month', 'after', 'months', 'of', 'testing', '.', 'Officials', 'said', 'the', 'service', 'will', 'reduce', 'traffic', 'and', 'improve', 'access', '.']
AFTER : ['city', 'council', 'announced', 'new', 'transit', 'line', 'open', 'next', 'month', 'months', 'testing', '.', 'Officials', 'said', 'service', 'reduce', 'traffic', 'improve', 'access', '.']
Vocabulary size before: 25
Vocabulary size after : 19
Vocabulary reduction   : 6

--- Cus

### New lines explained

- `stopwords.words('english')` retrieves NLTK's English stopword list.
- `set(...)` converts the list into a set, which is efficient for checking whether a word belongs to the stopword list.
- `token.lower()` converts only the token used for comparison to lowercase. This means `The`, `THE`, and `the` can all match the stopword `the`.
- The list comprehension `[token for token in tokens if ...]` creates a new list containing only tokens that are **not** stopwords.
- `set(token.lower() for token in tokens)` creates a vocabulary: the unique lowercase token forms.
- `len(...)` gives the vocabulary size.

### Why stopword removal matters

It can reduce noisy, high-frequency words and shrink the feature space for some NLP models. If we skip it, a simple frequency-based model may spend many features on common grammatical words. However, stopword removal is **task-dependent**: words such as `not` can be important for sentiment, so a real project should verify that its stopword list is appropriate.

## 8. Step 3 — Strip punctuation using Regex

Now we remove punctuation/symbol characters from the stopword-filtered tokens.

The Regex pattern below keeps word characters (`\w`) and whitespace (`\s`) and removes other characters. This is useful because punctuation can create unnecessary features such as `word`, `word!`, and `word?` when the downstream task does not need that distinction.

> **Code comment required by the assignment:** punctuation stripping can reduce noisy or duplicate-looking features and make later operations such as stemming, lemmatisation, counting, and vectorisation more consistent. It should not be used blindly when punctuation itself carries meaning, such as in sentiment, legal text, or social-media analysis.

In [7]:
def strip_punctuation(tokens):
    # Removing punctuation reduces noisy feature variants and makes downstream NLP processing more consistent.
    return [re.sub(r'[^\w\s]', '', token) for token in tokens]

without_punctuation = {}

for sample_name, tokens in no_stopwords.items():
    cleaned_tokens = strip_punctuation(tokens)
    cleaned_tokens = [token for token in cleaned_tokens if token]
    without_punctuation[sample_name] = cleaned_tokens

    print(f'--- {sample_name} ---')
    print('BEFORE:', tokens)
    print('AFTER :', cleaned_tokens)
    print()

--- Social Media ---
BEFORE: ['OMG', '!', '!', '!', 'LOVE', 'new', 'phone', 'update', '😍🔥', 'fast', ',', 'battery', 'drains', 'quickly', '...', '#', 'happy']
AFTER : ['OMG', 'LOVE', 'new', 'phone', 'update', 'fast', 'battery', 'drains', 'quickly', 'happy']

--- News Article ---
BEFORE: ['city', 'council', 'announced', 'new', 'transit', 'line', 'open', 'next', 'month', 'months', 'testing', '.', 'Officials', 'said', 'service', 'reduce', 'traffic', 'improve', 'access', '.']
AFTER : ['city', 'council', 'announced', 'new', 'transit', 'line', 'open', 'next', 'month', 'months', 'testing', 'Officials', 'said', 'service', 'reduce', 'traffic', 'improve', 'access']

--- Customer Review ---
BEFORE: ['room', 'clean', ',', 'staff', 'friendly', ',', 'breakfast', 'amazing', '!', 'would', 'definitely', 'stay', '😊', '.']
AFTER : ['room', 'clean', 'staff', 'friendly', 'breakfast', 'amazing', 'would', 'definitely', 'stay']



### Regex explained carefully

`re.sub(pattern, replacement, text)` searches for every part of `text` matching `pattern` and replaces it.

Here the pattern is `[^\w\s]`:
- `\w` means a word character.
- `\s` means whitespace.
- `[...]` defines a character class.
- `^` inside the character class means **not**.
- Therefore `[^\w\s]` means a character that is neither a word character nor whitespace.
- `''` is the replacement, so matching characters are deleted.

The final list comprehension removes empty strings created when a token consisted entirely of punctuation/symbols.

## 9. Step 4 — Stemming with PorterStemmer

Stemming tries to reduce related words to a common stem by applying rules to the word ending. The result is not necessarily a valid English word.

In [8]:
stemmer = PorterStemmer()
stemmed = {}

for sample_name, tokens in without_punctuation.items():
    stemmed_tokens = [stemmer.stem(token.lower()) for token in tokens]
    stemmed[sample_name] = stemmed_tokens

    print(f'--- {sample_name} ---')
    print('BEFORE:', tokens)
    print('AFTER :', stemmed_tokens)
    print()

--- Social Media ---
BEFORE: ['OMG', 'LOVE', 'new', 'phone', 'update', 'fast', 'battery', 'drains', 'quickly', 'happy']
AFTER : ['omg', 'love', 'new', 'phone', 'updat', 'fast', 'batteri', 'drain', 'quickli', 'happi']

--- News Article ---
BEFORE: ['city', 'council', 'announced', 'new', 'transit', 'line', 'open', 'next', 'month', 'months', 'testing', 'Officials', 'said', 'service', 'reduce', 'traffic', 'improve', 'access']
AFTER : ['citi', 'council', 'announc', 'new', 'transit', 'line', 'open', 'next', 'month', 'month', 'test', 'offici', 'said', 'servic', 'reduc', 'traffic', 'improv', 'access']

--- Customer Review ---
BEFORE: ['room', 'clean', 'staff', 'friendly', 'breakfast', 'amazing', 'would', 'definitely', 'stay']
AFTER : ['room', 'clean', 'staff', 'friendli', 'breakfast', 'amaz', 'would', 'definit', 'stay']



### New commands explained

- `PorterStemmer()` creates the stemming algorithm object.
- `stemmer.stem(token)` applies the Porter stemming rules to one word.
- `.lower()` makes the comparison/output case-consistent before stemming.

### Why stemming matters

Stemming can reduce the number of distinct word forms. For example, words such as `connected`, `connecting`, and `connection` may be reduced toward related stems. This can help traditional information-retrieval or classification systems.

### What breaks if we skip it?

A model that treats every inflected form as a completely separate feature may have a larger, sparser feature space and may need more data to learn that related word forms are connected. Stemming is not always desirable, though, because aggressive stemming can produce awkward non-words.

## 10. Step 5 — Lemmatization with WordNetLemmatizer

Lemmatization is different from stemming. Instead of simply cutting word endings, a lemmatiser tries to return a dictionary-style base form, called a **lemma**.

For a simple NLTK demonstration, `WordNetLemmatizer.lemmatize()` is used with its default noun setting. More advanced projects can provide part-of-speech tags so verbs and adjectives are lemmatised more accurately.

In [9]:
lemmatizer = WordNetLemmatizer()
lemmatized = {}

for sample_name, tokens in without_punctuation.items():
    lemmatized_tokens = [lemmatizer.lemmatize(token.lower()) for token in tokens]
    lemmatized[sample_name] = lemmatized_tokens

    print(f'--- {sample_name} ---')
    print('BEFORE:', tokens)
    print('AFTER :', lemmatized_tokens)
    print()

--- Social Media ---
BEFORE: ['OMG', 'LOVE', 'new', 'phone', 'update', 'fast', 'battery', 'drains', 'quickly', 'happy']
AFTER : ['omg', 'love', 'new', 'phone', 'update', 'fast', 'battery', 'drain', 'quickly', 'happy']

--- News Article ---
BEFORE: ['city', 'council', 'announced', 'new', 'transit', 'line', 'open', 'next', 'month', 'months', 'testing', 'Officials', 'said', 'service', 'reduce', 'traffic', 'improve', 'access']
AFTER : ['city', 'council', 'announced', 'new', 'transit', 'line', 'open', 'next', 'month', 'month', 'testing', 'official', 'said', 'service', 'reduce', 'traffic', 'improve', 'access']

--- Customer Review ---
BEFORE: ['room', 'clean', 'staff', 'friendly', 'breakfast', 'amazing', 'would', 'definitely', 'stay']
AFTER : ['room', 'clean', 'staff', 'friendly', 'breakfast', 'amazing', 'would', 'definitely', 'stay']



### Stemming vs lemmatization — the important difference

| Method | Main idea | Output quality |
|---|---|---|
| Stemming | Uses rules to cut/reduce word endings | Can produce non-words such as `studi` or `connect` |
| Lemmatization | Uses linguistic/dictionary information | Usually produces a valid base word |

Neither method is automatically better. Stemming is often faster and simpler; lemmatisation is usually more linguistically meaningful but may require more information, especially part-of-speech tags.

## 11. Side-by-side stemming vs lemmatisation comparison

The assignment asks us to apply both methods to the same cleaned input and show the differences side by side.

In [10]:
for sample_name in samples:
    print(f'=== {sample_name} ===')
    print(f"{'Original':<20} {'Stemmed':<20} {'Lemmatized':<20}")
    print('-' * 60)

    original_tokens = without_punctuation[sample_name]
    stemmed_tokens = stemmed[sample_name]
    lemmatized_tokens = lemmatized[sample_name]

    for original, stem, lemma in zip(original_tokens, stemmed_tokens, lemmatized_tokens):
        print(f'{original:<20} {stem:<20} {lemma:<20}')
    print()

=== Social Media ===
Original             Stemmed              Lemmatized          
------------------------------------------------------------
OMG                  omg                  omg                 
LOVE                 love                 love                
new                  new                  new                 
phone                phone                phone               
update               updat                update              
fast                 fast                 fast                
battery              batteri              battery             
drains               drain                drain               
quickly              quickli              quickly             
happy                happi                happy               

=== News Article ===
Original             Stemmed              Lemmatized          
------------------------------------------------------------
city                 citi                 city                
council         

### New command explained

`zip(original_tokens, stemmed_tokens, lemmatized_tokens)` combines the three lists position by position. This lets us print the same word next to its stemmed and lemmatised versions.

The format strings such as `f'{original:<20}'` are only for neat console alignment: `<20` means left-align the text in a field 20 characters wide.

## 12. Five-step before-and-after comparison across all samples

This section provides a compact comparison suitable for the final submission. The exact printed output will depend on the NLTK version and tokenisation behaviour on the machine where the notebook is run.

In [11]:
for sample_name in samples:
    print('=' * 90)
    print(sample_name)
    print('=' * 90)

    print('\nSTEP 1A — Sentence tokenisation')
    print('Before:', samples[sample_name]['text'])
    print('After :', sentence_tokens[sample_name])

    print('\nSTEP 1B — Word tokenisation')
    print('Before:', samples[sample_name]['text'])
    print('After :', word_tokens[sample_name])

    print('\nSTEP 2 — Stopword removal')
    print('Before:', word_tokens[sample_name])
    print('After :', no_stopwords[sample_name])

    print('\nSTEP 3 — Punctuation stripping')
    print('Before:', no_stopwords[sample_name])
    print('After :', without_punctuation[sample_name])

    print('\nSTEP 4 — Stemming')
    print('Before:', without_punctuation[sample_name])
    print('After :', stemmed[sample_name])

    print('\nSTEP 5 — Lemmatization')
    print('Before:', without_punctuation[sample_name])
    print('After :', lemmatized[sample_name])


Social Media

STEP 1A — Sentence tokenisation
Before: OMG!!! I LOVE this new phone update 😍🔥 It is SO fast now, but the battery drains quickly... #happy
After : ['OMG!!!', 'I LOVE this new phone update 😍🔥 It is SO fast now, but the battery drains quickly... #happy']

STEP 1B — Word tokenisation
Before: OMG!!! I LOVE this new phone update 😍🔥 It is SO fast now, but the battery drains quickly... #happy
After : ['OMG', '!', '!', '!', 'I', 'LOVE', 'this', 'new', 'phone', 'update', '😍🔥', 'It', 'is', 'SO', 'fast', 'now', ',', 'but', 'the', 'battery', 'drains', 'quickly', '...', '#', 'happy']

STEP 2 — Stopword removal
Before: ['OMG', '!', '!', '!', 'I', 'LOVE', 'this', 'new', 'phone', 'update', '😍🔥', 'It', 'is', 'SO', 'fast', 'now', ',', 'but', 'the', 'battery', 'drains', 'quickly', '...', '#', 'happy']
After : ['OMG', '!', '!', '!', 'LOVE', 'new', 'phone', 'update', '😍🔥', 'fast', ',', 'battery', 'drains', 'quickly', '...', '#', 'happy']

STEP 3 — Punctuation stripping
Before: ['OMG', '!', '!

## 13. Why every preprocessing step matters

### 1. Sentence and word tokenisation
**Purpose:** Convert continuous text into units that NLP algorithms can process.  
**If skipped:** Word counts, vocabulary creation, stopword removal, stemming, and lemmatisation become unreliable because the system has no clear token boundaries.

### 2. Stopword removal
**Purpose:** Optionally remove very common grammatical words and reduce the number of features.  
**If skipped:** High-frequency function words can dominate simple frequency-based representations and increase feature dimensionality.  
**Caution:** Do not automatically remove all stopwords for every task. Negation words can be important in sentiment analysis.

### 3. Punctuation stripping
**Purpose:** Remove punctuation/symbol variation when punctuation is not useful for the downstream task.  
**If skipped:** A model may treat `great`, `great!`, and `great?` as different features.  
**Caution:** Punctuation can itself carry meaning, so some NLP applications should preserve it.

### 4. Stemming
**Purpose:** Collapse related word forms into shorter stems.  
**If skipped:** Different inflected forms may remain separate features, increasing sparsity.  
**Caution:** Stems are not guaranteed to be real words.

### 5. Lemmatization
**Purpose:** Reduce words toward meaningful dictionary base forms.  
**If skipped:** Related grammatical forms can remain separate features.  
**Caution:** Lemmatization depends on linguistic information and can be slower/more complex than simple stemming.

## 14. Final conclusion

Text preprocessing converts messy natural language into a representation that is easier for NLP algorithms to analyse. In this project, tokenisation identified sentences and words, stopword removal reduced common-word noise, Regex removed unwanted punctuation/symbols, stemming reduced words to approximate stems, and WordNet lemmatisation produced more linguistically meaningful base forms.

The most important lesson is that preprocessing is **task-dependent**. More cleaning is not always better. A production NLP pipeline should test whether removing punctuation, stopwords, or word endings actually improves the downstream task.

## 15. Submission checklist

- [ ] Three raw text samples are included.
- [ ] The samples demonstrate punctuation, emojis, stopwords, and inconsistent casing.
- [ ] Sentence-level tokenisation is shown.
- [ ] Word-level tokenisation is shown.
- [ ] NLTK stopwords are removed.
- [ ] Vocabulary size is compared before and after stopword removal.
- [ ] Regex punctuation stripping is implemented and explained in a code comment.
- [ ] Porter stemming is applied.
- [ ] WordNet lemmatisation is applied.
- [ ] Stemming and lemmatisation are printed side by side.
- [ ] Before/after comparisons are shown for all samples.
- [ ] A short explanation describes why each step matters and what can go wrong if it is skipped.
- [ ] If required by the instructor, the three representative samples are replaced with real sourced excerpts and their source details are recorded.